# Kaggle dataset inspection — `cattle-weight-detection-model-dataset-12k`

**Run this notebook on Kaggle** (preferred — dataset is mounted automatically) or Colab with the dataset downloaded via `kaggle datasets download`.

Goal: answer the gating questions for our pipeline before writing any training code.

1. What is the actual on-disk folder structure?
2. What annotation files exist per batch (B2/B3/B4) and per view (Side/Rear)?
3. What format are the keypoint annotations (COCO JSON, CSV, …)?
4. **Do per-image weight (kg) labels exist? Where?**
5. Are there BCS, body length, withers height, heart girth, or hip length labels anywhere?
6. What are the **physical dimensions of the sticker** (the px↔cm reference)?
7. Can we recover an animal ID from filenames for group-aware CV?
8. Visual sanity check — do mask colors and keypoint coordinates match the documented schema?

The final cell prints a JSON summary; copy it back into the chat so we can lock down the schema for the parser.

Dataset: <https://www.kaggle.com/datasets/sadhliroomyprime/cattle-weight-detection-model-dataset-12k> (CC BY 4.0, Sadhli Roomy / Acme AI). Cite as required.

## Setup

On **Kaggle**: add the dataset to the notebook ("+ Add data" → search `cattle-weight-detection-model-dataset-12k`) and the default mount is `/kaggle/input/cattle-weight-detection-model-dataset-12k`.

On **Colab**: `pip install kaggle`, place your `~/.kaggle/kaggle.json`, then `kaggle datasets download -d sadhliroomyprime/cattle-weight-detection-model-dataset-12k --unzip -p /content/cattle_dataset`.

In [ ]:
import json
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

# --- Edit this if your mount path differs ---
CANDIDATE_ROOTS = [
    "/kaggle/input/cattle-weight-detection-model-dataset-12k",
    "/content/cattle_dataset",
    "./cattle_dataset",
]
DATASET_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.isdir(p)), None)
assert DATASET_ROOT, (
    f"Could not find dataset at any of {CANDIDATE_ROOTS}. "
    "Edit CANDIDATE_ROOTS to point at the unzipped dataset directory."
)
DATASET_ROOT = Path(DATASET_ROOT)
print("Using dataset root:", DATASET_ROOT)

# Findings accumulator — printed as JSON at the end.
findings = {"dataset_root": str(DATASET_ROOT)}

## 1. Top-level folder structure

Confirms the structure described in the dataset card: `Pixel/{B2,B3,B4}/...` (annotations + images) and `Vector/{B2,B3,B4}/...` (vector / keypoint data).

In [ ]:
def walk_tree(root: Path, max_depth: int = 4) -> dict:
    """Return a nested dict {name: subtree_or_filecount}."""
    def _walk(path: Path, depth: int):
        if depth > max_depth:
            return "..."
        if path.is_file():
            return None
        out = {}
        try:
            entries = sorted(path.iterdir())
        except PermissionError:
            return "<denied>"
        # Group leaf files by extension counts; recurse into directories.
        files_by_ext = Counter()
        for entry in entries:
            if entry.is_file():
                files_by_ext[entry.suffix.lower() or "<noext>"] += 1
            else:
                out[entry.name + "/"] = _walk(entry, depth + 1)
        for ext, n in sorted(files_by_ext.items()):
            out[f"*{ext}"] = n
        return out
    return {root.name: _walk(root, 0)}

structure = walk_tree(DATASET_ROOT)
print(json.dumps(structure, indent=2)[:6000])  # truncate if very wide
findings["structure_sample"] = structure

## 2. Annotation file inventory

List every annotation-looking file (json/csv/txt/xml) with its size, so we know what to open next.

In [ ]:
ANN_SUFFIXES = {".json", ".csv", ".txt", ".xml", ".xlsx", ".xls"}
annotation_files = []
for path in DATASET_ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in ANN_SUFFIXES:
        annotation_files.append({
            "path": str(path.relative_to(DATASET_ROOT)),
            "size_bytes": path.stat().st_size,
        })
annotation_files.sort(key=lambda r: r["path"])
print(f"Found {len(annotation_files)} annotation files.")
for row in annotation_files[:40]:
    print(f"  {row['size_bytes']:>10} B   {row['path']}")
if len(annotation_files) > 40:
    print(f"  ... ({len(annotation_files) - 40} more)")
findings["annotation_files"] = annotation_files

## 3. Peek inside one annotation file per batch

Just enough to identify the format (COCO JSON vs flat CSV vs custom).

In [ ]:
def peek_file(path: Path, max_chars: int = 1500) -> dict:
    info = {"path": str(path.relative_to(DATASET_ROOT))}
    try:
        if path.suffix.lower() == ".json":
            with path.open("r", encoding="utf-8") as f:
                data = json.load(f)
            info["json_top_keys"] = list(data.keys()) if isinstance(data, dict) else "<list>"
            if isinstance(data, dict) and "categories" in data:
                info["categories"] = data["categories"]
                info["image_count"] = len(data.get("images", []))
                info["annotation_count"] = len(data.get("annotations", []))
                if data.get("annotations"):
                    info["sample_annotation"] = data["annotations"][0]
                if data.get("images"):
                    info["sample_image"] = data["images"][0]
        else:
            with path.open("r", encoding="utf-8", errors="replace") as f:
                head = f.read(max_chars)
            info["head"] = head
    except Exception as exc:
        info["error"] = repr(exc)
    return info

# Pick one annotation file per top-level directory under DATASET_ROOT/Pixel and /Vector.
picked = {}
for row in annotation_files:
    parts = Path(row["path"]).parts
    bucket = "/".join(parts[:3])  # e.g. Pixel/B2/Back
    picked.setdefault(bucket, row["path"])
    if len(picked) >= 12:
        break

peeks = []
for bucket, rel in picked.items():
    peeks.append({"bucket": bucket, **peek_file(DATASET_ROOT / rel)})

for p in peeks:
    print("=" * 80)
    print(p["bucket"], "->", p["path"])
    for k, v in p.items():
        if k in ("bucket", "path"):
            continue
        val = json.dumps(v, indent=2) if not isinstance(v, str) else v
        print(f"  {k}:\n{val[:1500]}")
findings["annotation_peeks"] = peeks

## 4. Search for weight / BCS / morphometric labels

Greps every annotation file for tell-tale keys.

In [ ]:
KEYWORDS = [
    "weight", "kg", "mass",
    "bcs", "body_condition", "condition_score",
    "length", "height", "girth", "hip", "withers",
    "sticker", "reference", "scale",
]
hits = defaultdict(list)
for row in annotation_files:
    path = DATASET_ROOT / row["path"]
    if path.stat().st_size > 50 * 1024 * 1024:  # skip giant files for the grep
        continue
    try:
        with path.open("r", encoding="utf-8", errors="replace") as f:
            text = f.read().lower()
    except Exception:
        continue
    for kw in KEYWORDS:
        if kw in text:
            hits[kw].append(row["path"])

for kw, files in sorted(hits.items()):
    print(f"{kw!r}: {len(files)} files")
    for f in files[:5]:
        print(f"    {f}")
    if len(files) > 5:
        print(f"    ... ({len(files) - 5} more)")
findings["keyword_hits"] = {kw: files for kw, files in hits.items()}

## 5. COCO categories + keypoint schema (if applicable)

If the Vector annotations are COCO-format JSON, pull category names and `keypoints` lists so we can hard-code the schema in the parser.

In [ ]:
coco_schemas = []
for row in annotation_files:
    if not row["path"].lower().endswith(".json"):
        continue
    path = DATASET_ROOT / row["path"]
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        continue
    if not isinstance(data, dict) or "categories" not in data:
        continue
    schema = {
        "path": row["path"],
        "categories": data["categories"],
        "n_images": len(data.get("images", [])),
        "n_annotations": len(data.get("annotations", [])),
    }
    coco_schemas.append(schema)

for s in coco_schemas[:6]:
    print("=" * 60)
    print(s["path"], f"({s['n_images']} images, {s['n_annotations']} anns)")
    print(json.dumps(s["categories"], indent=2))
findings["coco_schemas"] = coco_schemas[:6]

## 6. Sticker physical dimensions

Search any README / documentation files in the dataset root for the sticker's real-world size. This is critical — without it, our px↔cm calibration is unanchored.

In [ ]:
DOC_NAMES = ("readme", "info", "description", "notes", "about")
DOC_SUFFIXES = {".md", ".txt", ".pdf", ".docx"}
doc_files = []
for path in DATASET_ROOT.rglob("*"):
    if not path.is_file():
        continue
    name = path.stem.lower()
    if any(d in name for d in DOC_NAMES) or path.suffix.lower() in DOC_SUFFIXES:
        doc_files.append(str(path.relative_to(DATASET_ROOT)))

print("Doc-like files:", doc_files)

STICKER_PATTERN = re.compile(
    r"sticker[^.\n]{0,200}?(\d+(?:\.\d+)?\s*(?:cm|mm|inch|in))", re.I
)
for rel in doc_files:
    path = DATASET_ROOT / rel
    if path.suffix.lower() not in {".md", ".txt"}:
        continue
    try:
        text = path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        continue
    for m in STICKER_PATTERN.finditer(text):
        print(f"  [{rel}] match: ...{text[max(0, m.start()-40):m.end()+40]}...")
findings["doc_files"] = doc_files
print("\nIf nothing matched: check the Kaggle dataset description (web UI) or the original\n"
      "Acme AI / TensorFlow forum post for the sticker dimension.")

## 7. Animal ID from filenames

The dataset description mentions filename conventions vary across batches (e.g. `B2/B3: <id>/<view>`, `B4: <id>-<view>/...`). Sample filenames and try to extract an animal-level identifier.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}
samples_per_batch = defaultdict(list)
for path in DATASET_ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in IMAGE_SUFFIXES:
        continue
    parts = path.relative_to(DATASET_ROOT).parts
    # Batch is the second component for Pixel/Vector subtrees.
    if len(parts) >= 2:
        batch = "/".join(parts[:3])
        if len(samples_per_batch[batch]) < 8:
            samples_per_batch[batch].append(path.name)

for batch, names in sorted(samples_per_batch.items()):
    print("=" * 60)
    print(batch)
    for n in names:
        print("  ", n)
findings["sample_filenames"] = dict(samples_per_batch)

## 8. Visual sanity check

Render one side-view image with its mask + keypoints overlaid. Confirms the documented mask color codes are what's actually in the files.

In [ ]:
# Best-effort: find the first side-view image with a matching mask under Pixel/.
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

side_images = []
for path in DATASET_ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in IMAGE_SUFFIXES:
        continue
    parts = [p.lower() for p in path.relative_to(DATASET_ROOT).parts]
    if "side" in parts and "images" in parts:
        side_images.append(path)
        if len(side_images) >= 6:
            break

print(f"Found {len(side_images)} side-view image candidates.")
if side_images:
    image_path = side_images[0]
    print("Rendering:", image_path.relative_to(DATASET_ROOT))
    img = np.array(Image.open(image_path).convert("RGB"))
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(img)
    ax.set_title(image_path.name)
    ax.axis("off")
    plt.show()
    print("Image shape:", img.shape, "dtype:", img.dtype)
    # Try to find the paired mask by swapping 'images' -> 'annotations'.
    rel = image_path.relative_to(DATASET_ROOT)
    mask_candidate = DATASET_ROOT / Path(*["annotations" if p == "images" else p for p in rel.parts])
    mask_candidate = mask_candidate.with_suffix(".png")
    if mask_candidate.exists():
        mask = np.array(Image.open(mask_candidate).convert("RGB"))
        print("Mask shape:", mask.shape)
        unique_colors = {tuple(c) for c in mask.reshape(-1, 3).tolist()[::1000]}  # sparse sample
        print("Sampled unique mask colors:", sorted(unique_colors)[:10])
        findings["sample_mask"] = {
            "path": str(mask_candidate.relative_to(DATASET_ROOT)),
            "unique_colors_sampled": [list(c) for c in sorted(unique_colors)[:10]],
        }
    else:
        print("No paired mask at:", mask_candidate.relative_to(DATASET_ROOT))

## 9. Findings summary — copy this back to chat

Paste the printed JSON (or just the most important fields) into the conversation so we can lock down the parser.

In [ ]:
# Drop huge nested fields before printing.
summary = dict(findings)
summary.pop("structure_sample", None)  # keep separate; printed earlier
summary.pop("sample_filenames", None)
# Trim per-keyword hit lists to the first 5.
if "keyword_hits" in summary:
    summary["keyword_hits"] = {k: v[:5] for k, v in summary["keyword_hits"].items()}
print(json.dumps(summary, indent=2)[:8000])